In [ ]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

epochs = 25
batch_size = 100
learning_rate = 0.001

CIFAR100_MEAN = [0.5071, 0.4865, 0.4409]
CIFAR100_STD = [0.2673, 0.2564, 0.2762]
DATA_ROOT = './cifar100_data'

train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)
])

train_dataset = datasets.CIFAR100(root=DATA_ROOT, train=True, transform=train_transforms, download=True)
test_dataset = datasets.CIFAR100(root=DATA_ROOT, train=False, transform=test_transforms)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
def create_conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0):
    return nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride,
                     padding=padding, bias=False)

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate=32):
        super().__init__()
        squeezed_channels = 4 * growth_rate
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = create_conv2d(in_channels=in_channels, out_channels=squeezed_channels, kernel_size=1)
        self.bn2 = nn.BatchNorm2d(squeezed_channels)
        self.conv2 = create_conv2d(in_channels=squeezed_channels, out_channels=growth_rate, kernel_size=3,
                                   padding=1)

    def forward(self, x):
        out = self.bn1(x)
        out = self.relu(out)
        out = self.conv1(out)

        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv2(out)

        return torch.concat((x, out), dim=1)


class TransitionLayer(nn.Module):
    def __init__(self, in_channels, compression_rate=0.5):
        super().__init__()
        out_channels = int(in_channels * compression_rate)
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv = create_conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        self.avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        out = self.bn(x)
        out = self.relu(out)
        out = self.conv(out)
        out = self.avg_pool(out)

        return out

In [ ]:
class DenseNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super().__init__()
        self.in_channels = 64
        self.conv = create_conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3)
        self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.db1 = self.create_dense_blocks(DenseLayer, layers[0])
        self.db2 = self.create_dense_blocks(DenseLayer, layers[1])
        self.db3 = self.create_dense_blocks(DenseLayer, layers[2])
        self.db4 = self.create_dense_blocks(DenseLayer, layers[3], is_transition_applied=False)
        self.bn = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1024, num_classes)

    def create_dense_blocks(self, block, blocks, growth_rate=32, is_transition_applied=True):
        layers = []
        for i in range(blocks):
            layers.append(block(self.in_channels, growth_rate))
            self.in_channels += growth_rate

        if is_transition_applied:
            return nn.Sequential(*layers, self.create_transition_layer(TransitionLayer))
        else:
            return nn.Sequential(*layers)

    def create_transition_layer(self, block, compression_rate=0.5):
        layer = block(self.in_channels)
        self.in_channels = int(self.in_channels * compression_rate)

        return layer

    def forward(self, x):
        out = self.conv(x)
        out = self.max_pool(out)

        out = self.db1(out)
        out = self.db2(out)
        out = self.db3(out)
        out = self.db4(out)

        out = self.bn(out)
        out = self.relu(out)
        out = self.avg_pool(out).view(out.size(0), -1)
        out = self.fc(out)

        return out